<a href="https://colab.research.google.com/github/gyeongee/TIL/blob/main/%EC%9E%A0%EC%9E%AC_%EC%9A%94%EC%9D%B8_%ED%98%91%EC%97%85_%ED%95%84%ED%84%B0%EB%A7%81ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

행렬 분해를 이용한 잠재 요인 협업 필터링을 구현해 보겠습니다. 행렬 분해 잠재 요인 협업 필터링은 SVD나 NMF 등을 적용할 수 있는데, 일반적으로 행렬 분해에는 SVD가 자주 사용되지만 사용자-아이템 평점 행렬에는 사용자가 평점을 매기지 않은 널 데이터가 많기 때문에 주로 SGD나 ALS 기반의 행렬 분해를 이용합니다.

여기서는 SGD 기반의 행렬 분해를 구현하고 이를 기반으로 사용자에게 영화를 추천해 보겠습니다.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
movies=pd.read_csv('/content/movie.csv')
ratings=pd.read_csv('/content/rating.csv')


In [3]:

ratings=ratings[['userId','movieId','rating']]
ratings_matrix=ratings.pivot_table('rating',index='userId',columns='movieId')

# title 칼럼을 얻기 위해 movies와 조인 수행
rating_movies=pd.merge(ratings,movies,on='movieId')

#columns='title'로 title 칼럼으로 pivot 수행
ratings_matrix=rating_movies.pivot_table('rating',index='userId',columns='title')
# Nan 값을 0으로 채우기
ratings_matrix=ratings_matrix.fillna(0)


In [4]:
from sklearn.metrics import mean_squared_error

# 사용자가 평점을 부여한 영화에 대해서만 예측 성능 평가 MSE를 구함
def get_rmse(R,P,Q,non_zeros):
  error=0
  # 두 개의 분해된 행렬 P와 Q.T의 내적으로 예측 R 행렬 생성
  full_pred_matrix=np.dot(P,Q.T)

  # 실제 R 행렬에서 널이 아닌 위치 인덱스 추출해 실제 R 행렬과 예측 행렬의 RMSE 추출
  x_non_zero_ind=[non_zero[0] for non_zero in non_zeros]
  y_non_zero_ind=[non_zero[1] for non_zero in non_zeros]
  R_non_zeros=R[x_non_zero_ind,y_non_zero_ind]
  full_pred_matrix_non_zeros=full_pred_matrix[x_non_zero_ind,y_non_zero_ind]

  mse=mean_squared_error(R_non_zeros,full_pred_matrix_non_zeros)
  rmse=np.sqrt(mse)

  return rmse

Q 업데이트에 이미 수정된 `P[i,:]`을 사용하고 있으므로 같은 시점의 `P,Q`를 기준으로 업데이트하는 방식으로 구현하겠습니다.

In [9]:
# 행렬 분해 로직_matrix_factorization()
# R : 원본 사용자-아이템 평점 행렬
# K : 잠재 요인
# steps : SGD의 반복 횟수
# r_lambda : L2 규제 계수

def matrix_factorization(R,K,steps=200,learning_rate=0.01,r_lambda=0.01):
  num_users,num_items=R.shape
  # p와 Q 매트릭스의 크기를 지정하고 정규분포를 가진 랜덤한 값으로 입력합니다.
  P=np.random.normal(scale=1./K,size=(num_users,K))
  Q=np.random.normal(scale=1./K,size=(num_items,K))

  # R>0인 행 위치,열 위치,값을 non_zeros 리스트 객체에 저장
  non_zeros=[(i,j,R[i,j])for i in range(num_users)for j in range(num_items)if R[i,j]>0]

  # SGD기법으로 P와 Q 매트릭스를 계속 업데이트
  for step in range(steps):
    for i,j,r in non_zeros:
      # 실제 값과 예측 값의 차이인 오류 값 구함
      eij=r - np.dot(P[i,:],Q[j,:].T)

      ### 코드 수정 ###

      # 기존 P와 Q를 복사
      p = P[i,:].copy()
      q = Q[j,:].copy()
      # Regularization을 반영한 SGD 업데이트 공식 적용
      P[i,:] = p + learning_rate * (eij * q - r_lambda * p)
      Q[j,:] = q + learning_rate * (eij * p - r_lambda * q)

    rmse=get_rmse(R,P,Q,non_zeros)
    if (step % 10)==0:
      print(f'### interation step: {step}, rmse:{rmse}')

  return P,Q


In [7]:
print("ratings_matrix NaN:", np.isnan(ratings_matrix.values).sum())
print("ratings_matrix Inf:", np.isinf(ratings_matrix.values).sum())

ratings_matrix NaN: 0
ratings_matrix Inf: 0


In [8]:
P, Q = matrix_factorization(
    ratings_matrix.values,
    K=50,
    steps=200,
    learning_rate=0.01,
    r_lambda=0.01
)

print("P NaN:", np.isnan(P).sum())
print("Q NaN:", np.isnan(Q).sum())
print("P Inf:", np.isinf(P).sum())
print("Q Inf:", np.isinf(Q).sum())

### interation step: 0, rmse:3.4464285910012107


ValueError: Input contains NaN.

In [10]:
# 사용자-아이템 평점 행렬을 행렬 분해
P,Q = matrix_factorization(ratings_matrix.values,K=50,steps=200,learning_rate=0.01,
                           r_lambda=0.01)
pred_matrix=np.dot(P,Q.T)

### interation step: 0, rmse:3.086695162915319
### interation step: 10, rmse:0.7202477436250924
### interation step: 20, rmse:0.5887497821792622
### interation step: 30, rmse:0.5371443329012644
### interation step: 40, rmse:0.5123060531666206
### interation step: 50, rmse:0.4981917593607387
### interation step: 60, rmse:0.48915373605259954
### interation step: 70, rmse:0.4828580200742742
### interation step: 80, rmse:0.4782059305237759
### interation step: 90, rmse:0.47461950084602506
### interation step: 100, rmse:0.4717656586971181
### interation step: 110, rmse:0.46943834409064433
### interation step: 120, rmse:0.46750286448924133
### interation step: 130, rmse:0.4658672472080715
### interation step: 140, rmse:0.46446641747513817
### interation step: 150, rmse:0.4632529547379759
### interation step: 160, rmse:0.46219143759732656
### interation step: 170, rmse:0.46125484672213035
### interation step: 180, rmse:0.4604221983313584
### interation step: 190, rmse:0.45967694067762593


SGD를 반복하면서 P 또는 Q에 NaN이 생긴 것으로 보는 게 맞습니다.


더 쉽게 영화 아이템 칼럼을 이해하기 위해 반환된 예측 사용자-아이템 평점 행렬을 영화 타이틀을 칼럼명으로 가지는 DataFrame으로 변경하겠습니다.

In [11]:
ratings_pred_matrix = pd.DataFrame(data=pred_matrix,index=ratings_matrix.index,
                                   columns=ratings_matrix.columns)
ratings_pred_matrix.head(3)

title,$5 a Day (2008),'71 (2014),'Hellboy': The Seeds of Creation (2004),'Neath the Arizona Skies (1934),'R Xmas (2001),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),"'burbs, The (1989)",'night Mother (1986),...,[REC]² (2009),[REC]³ 3 Génesis (2012),"\\""Great Performances\""\"" Cats (1998)""",eXistenZ (1999),loudQUIETloud: A Film About the Pixies (2006),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nos amours (1983),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,3.487032,2.271403,3.099338,3.362530,2.537899,4.162950,3.004817,3.265682,3.309713,3.171863,...,3.826222,0.819546,3.298914,3.502670,3.817992,3.502308,3.493809,3.467687,0.851227,3.645006
2,3.840338,2.295983,3.588203,3.302965,2.242097,3.542203,2.783305,2.799557,3.452796,3.188586,...,3.646505,0.857467,2.586867,3.842649,3.951755,2.077098,3.081204,2.464717,0.757630,3.144561
3,3.276105,2.454197,3.063252,3.357686,2.428717,3.815354,3.020726,1.366911,2.282245,2.702855,...,3.595696,1.036993,3.247671,3.412539,4.291406,3.951511,1.917636,3.031011,0.987431,3.889885


In [16]:
def get_unseen_movies(ratings_matrix,userId):
  # userId로 입력받은 사용자의 모든 영화 정보를 추출해 Series로 반환
  # 반횐된 user_rating은 영화명을 인덱스로 가지는 Series 객체임
  user_rating=ratings_matrix.loc[userId,:]

  # user_rating이 0보다 크면 기존에 관람한 영화임. 대상 인덱스를 추출해 list 객체로 만듦
  already_seen=user_rating[user_rating > 0].index.tolist()

  # 모든 영화명을 list 객체로 만듦
  movies_list = ratings_matrix.columns.tolist()

  # list comprehension으로 already_seen에 해당하는 영화는 movies_list에서 제외함
  unseen_list = [movie for movie in movies_list if movie not in already_seen]

  return unseen_list

def recomm_movie_by_userid(pred_df,userId,unseen_list,top_n=10):
  # 예측 평점 DataFrame에서 사용자 id 인덱스와 unseen_list로 들어온 영화명 칼럼을 추출해
  # 가장 예측 평점이 높은 순으로 정렬함
  recomm_movies=pred_df.loc[userId,unseen_list].sort_values(ascending=False)[:top_n]
  return recomm_movies



In [17]:
# 예측 사용자-아이템 평점 행렬 정보를 이용해 개인화된 영화 추천
# 사용자가 관람하지 않은 영화명 추출
unseen_list=get_unseen_movies(ratings_matrix,9)

# 잠재 요인 협업 필터일으로 영화 추천
recomm_movies=recomm_movie_by_userid(ratings_pred_matrix,9,unseen_list,top_n=10)

# 평점 데이터를 DataFrame으로 생성
recomm_movies=pd.DataFrame(data=recomm_movies.values,index=recomm_movies.index,
                           columns=['pred_score'])
recomm_movies

,pred_score
title,
Very Bad Things (1998),6.757852
"Mission, The (1986)",6.168535
Natural Born Killers (1994),6.010236
Across the Universe (2007),5.933611
High Tension (Haute tension) (Switchblade Romance) (2003),5.933575
"Fisher King, The (1991)",5.793545
Jacob's Ladder (1990),5.712512
BASEketball (1998),5.662337
True Romance (1993),5.650011


앞 절의 아이템 기반 협업 필터일 결과와는 추천된 영화가 많이 다른 것을 알 수 있씁니다.